# Pseudonymization and Cryptographic Protection of Microdata

Double-hash pseudonymization (RIPEMD-160 + SHA-512), brute-force and dictionary attacks against it, a salted fix, encryption of a sensitive attribute with RSA, ElGamal and A5/1, and an independence (fairness) check on a predicted attribute.

Scenario: the same HR datasets as in notebook 01 (`workers.csv`, `absenteeism.csv`); this time identifiers are pseudonymized instead of removed, and sensitive values are encrypted.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import ast
import pandas as pd
path ="/content/drive/MyDrive/practica_2_privacidad_seguridad_Datos/"
df_workers = pd.read_csv(path+"data/workers.csv", sep = ';')
df_workers.info()
df_workers_original = df_workers.copy()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            22 non-null     int64 
 1   name          22 non-null     object
 2   sex           22 non-null     object
 3   age           22 non-null     int64 
 4   zip_code      22 non-null     int64 
 5   num_absences  22 non-null     int64 
 6   absences      22 non-null     object
 7   insurance     22 non-null     int64 
dtypes: int64(5), object(3)
memory usage: 1.5+ KB


In [3]:
df_absenteeism = pd.read_csv(path+"data/absenteeism.csv", sep = ';')
df_absenteeism.info()
df_absenteeism_original = df_absenteeism.copy()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 471 entries, 0 to 470
Data columns (total 21 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   ID                               470 non-null    float64
 1   Reason_for_absence               471 non-null    int64  
 2   Month_of_absence                 470 non-null    float64
 3   Day_of_the_week                  470 non-null    float64
 4   Seasons                          470 non-null    float64
 5   Transportation_expense           470 non-null    float64
 6   Distance_from_Residence_to_Work  470 non-null    float64
 7   Service_time                     470 non-null    float64
 8   Age                              470 non-null    float64
 9   Work_load_Average_day            470 non-null    float64
 10  Hit_target                       470 non-null    float64
 11  Disciplinary_failure             470 non-null    float64
 12  Education             

**Exercise 1:** Pseudonymize the identifier attributes by replacing each value with its RIPEMD-160 hash and then hashing that result with SHA-512.

**1.1** — Identify which attributes of `df_workers` act as identifiers.

In [ ]:
# Answer
#id: This is a unique identifier for each record in the table.
#name: The person's name is a direct identifier.
#zip_code: The zip code can be used to identify a person if it is combined with other attributes.

**1.2** — Implement a RIPEMD-160 hex-digest function.

In [5]:
!pip install pycryptodome

In [6]:
from Crypto.Hash import RIPEMD160           # Imports the RIPEMD160 library to compute the RIPEMD-160 hash
import hashlib                              # Imports the hashlib library to compute hashes with several algorithms

In [7]:
def ripemd160_hash(text):                   # Defines a function to compute the RIPEMD-160 hash of a text
    h = RIPEMD160.new()                     # Creates a new RIPEMD-160 hash object
    h.update(text.encode('utf-8'))          # Updates the hash object with the UTF-8 encoded text
    return h.hexdigest()                    # Returns the computed hash in hexadecimal format

**1.3** — Chain a SHA-512 hex digest on top of the RIPEMD-160 output.

In [8]:
def sha512_hash(text):                      # Defines a function to compute the SHA-512 hash of a text
    h = hashlib.sha512()                    # Creates a new SHA-512 hash object
    h.update(text.encode('utf-8'))          # Updates the hash object with the UTF-8 encoded text
    return h.hexdigest()                    # Returns the computed hash in hexadecimal format

**1.4** — Combine both hashes into a single `double_hash` function.

In [9]:
# Answer

def double_hash(text):                      # Defines a function to apply two consecutive hashes to a text
    ripemd160_result = ripemd160_hash(text) # Computes the RIPEMD-160 hash of the text
    sha512_result = sha512_hash(ripemd160_result) # Computes the SHA-512 hash of the previous result
    return sha512_result                    # Returns the final result of the double hash

**1.5** — Apply the pseudonymization to the dataframe and discuss why the process is needed.

In [10]:
# Answer

# We pseudonymize the identifiers
def pseudonymize_dataframe(df):             # Defines a function to pseudonymize identifiers in a DataFrame
    df['id'] = df['id'].astype(str).apply(double_hash)   # Applies the double hash to the 'id' column after converting it to string
    df['name'] = df['name'].apply(double_hash)           # Applies the double hash to the 'name' column
    df['zip_code'] = df['zip_code'].astype(str).apply(double_hash) # Applies the double hash to the 'zip_code' column after converting it to string
    return df                                 # Returns the DataFrame with the pseudonymized identifiers

# We apply pseudonymization
pseudonymized_workers_df = pseudonymize_dataframe(df_workers_original) # Calls the function to pseudonymize the original DataFrame
print(pseudonymized_workers_df)            # Prints the pseudonymized DataFrame

                                                   id  \
0   ceb51fb0fd5af1a95fc834731e16fe8fb8ad1ae4457042...   
1   06323d7afcbefa79a1e62f6bce86190dc83c11ed7334ac...   
2   a68d955d2c7dade51bd2c94385b9e772bbb939e36f95b3...   
3   81b645473612de18789ab375b11a27193b4c238ec361ce...   
4   28a46e4cef69414806eeb28d32c477f2ef1ecbe4cb1fb6...   
5   6bdf137e846390a3c5b58ed6962aa95a21afc4b240fc05...   
6   ad1a065aa70eeb69ccccf9170d06570f389435bf2b13e5...   
7   b2578d881e20c187b8df9d8e98a58041035a282a4eb8fe...   
8   98204a99b0b13f66ec431754ecd7fab07186e1ea6fde45...   
9   70a7a9b92c74803050a9f46a68ae929aed01ec25b3f036...   
10  02825ce71d3182e9c55fd0204f38a5075190cbfe538dc6...   
11  46d1cb3a25b4ea17fd877d08706a7efdddb2507655cd0e...   
12  27110cad9c225334272d8bcd0d725aa059db2bdb1a16f6...   
13  0a827fefb08c218cc0609b4f3202a3a6952c5538b9156b...   
14  0d315b61de345592dd9e6df818909d613a9779a26a9d24...   
15  becdb3e0dab4eb7b0dc657473fc8c7194bebe711e11ba0...   
16  f360f62f443536a5d8015317f76

<div class="alert alert-block alert-success">
Justification:

Pseudonymization is an essential technique in the protection of personal data, especially in contexts where sensitive information is handled, such as in the Human Resources department of the public administration. The pseudonymization process involves replacing direct identifiers with values derived by means of hash functions. This has several advantages:

- By transforming the identifiers into hash values, the direct identification of individuals is prevented, thus protecting their privacy.
In many countries, data protection legislation (such as the GDPR in Europe) requires the anonymization or pseudonymization of personal data to minimize risks in the event of data breaches.

- In the event of unauthorized access to the data, the pseudonymized information will not easily reveal the identity of the individuals, reducing the risk of misuse.

Knowing this:

The attributes id, name and zip_code were identified as direct identifiers that need to be pseudonymized.
A combination of RIPEMD-160 and SHA-512 was used to obtain robust hash values. RIPEMD-160 provides a good initial mixing and SHA-512 ensures high resistance against collision attacks.
The hash values were applied to the selected identifiers, replacing the original data with their corresponding hash values.


Although the current process uses two hash functions to maximize security, there are ways to optimize the process:

- Using only SHA-512 may be sufficiently secure and more efficient. SHA-512 is a robust algorithm on its own and removes the need to apply two hash functions.

- For large volumes of data, parallel or batch processing can be applied to improve efficiency. Using libraries such as multiprocessing in Python can significantly speed up the process.

- If certain identifiers are expected to be repeated, caching intermediate results can avoid redundant computations, improving efficiency.



</div>

**Exercise 2:** Attack the pseudonymized dataset knowing one person's full name — **2.1** without knowing which hash functions were used, and **2.2** knowing them — timing both attempts.

In [11]:
nombreConocido = "Honora Garbutt"

In [12]:
import itertools                                  # Imports the itertools library to handle iterations
import time                                       # Imports the time library to measure the execution time

# Start of the execution time
start_time = time.time()                          # Stores the current time to measure the execution time

# Simulation of a brute-force attack
# NOTE: This method will not be successful without knowledge of the hash and is only a theoretical example
for id_hash, name_hash, age, zip_code, absences in zip(    # Iterates over the original values of the DataFrame
        df_workers_original['id'], df_workers_original['name'], df_workers_original['age'],
        df_workers_original['zip_code'], df_workers_original['absences']):

    # This is simply a check; in reality, we need to decrypt the hash,
    # which is not feasible without knowledge of the hash algorithm
    if nombreConocido in name_hash:                # Checks whether the known name is in the name hash
        print(f"ID Hash: {id_hash}")               # Prints the ID hash
        print(f"Name Hash: {name_hash}")           # Prints the name hash
        print(f"Age: {age}")                       # Prints the age
        print(f"Zip Code Hash: {zip_code}")        # Prints the zip code hash
        print(f"Absences: {absences}")             # Prints the absences

# End of the execution time
end_time = time.time()                             # Stores the current time when the execution finishes
execution_time = end_time - start_time             # Computes the execution time by subtracting the start time

print(f"Execution time: {execution_time} seconds") # Prints the execution time


Execution time: 0.0008578300476074219 seconds


In [13]:
# Answer
# Start of the execution time

start_time = time.time()                          # Stores the current time to measure the execution time

# We generate the double hash of the known name
nombre_conocido_hash = double_hash(nombreConocido) # Computes the double hash of the known name

# We look for the hash in the pseudonymized DataFrame
matched_row = pseudonymized_workers_df[pseudonymized_workers_df['name'] == nombre_conocido_hash] # Looks for the hash in the pseudonymized DataFrame

result_2_2 = None                                 # Initializes the variable to store the result

# If we find a match, we show the information
if not matched_row.empty:                         # Checks whether a match was found
    result_2_2 = matched_row.to_dict('records')[0]# Converts the matching row into a dictionary and stores it in result_2_2

# End of the execution time
end_time = time.time()                            # Stores the current time when the execution finishes
execution_time_2_2 = end_time - start_time        # Computes the execution time by subtracting the start time

result_2_2, execution_time_2_2                    # Shows the result and the execution time

({'id': '0d315b61de345592dd9e6df818909d613a9779a26a9d242b41076222996c70945ce127fd829f0ad1a4c1aa3158f64f73fde1027b0b7b49309ee47d9419dc3dcd',
  'name': '2309b71c489923706bd137119e2ec2c589e14a22d942a88cbe913c282d3fc2cf339dae75180174d0aeb24f69f5ab138fe59a789dd28af06b033680848aa7b6de',
  'sex': 'F',
  'age': 43,
  'zip_code': '458aba1e7615af32248bd800e2ef90198b4e52dba4be7b2ca2bf1f8fd2015be70d299c30d32a49e88aa4e26dbe248e2dfedb2e53cd25b71a15ac072a73b0ab46',
  'num_absences': 37,
  'absences': '[5-9-241476,2-9-241476,5-10-253465,3-10-253465,5-11-306345,2-11-306345,5-11-306345,4-3-343253,4-4-326452,6-4-326452,6-4-326452,5-6-377550,3-9-294217,6-9-294217,6-9-294217,2-9-294217,3-10-265017,3-10-265017,2-11-284031,6-11-284031,4-1-330061,3-2-251818,2-6-253957,5-7-230290,5-8-249797,5-8-249797,5-9-261756,4-10-284853,6-10-284853,4-10-284853,4-11-268519,3-11-268519,4-12-280549,5-5-237656,4-5-237656,5-5-237656,5-6-275089]',
  'insurance': 1},
 0.012212038040161133)

<div class="alert alert-block alert-success">
Justification:


In this case, we try to recover information without knowing the hash functions used. It is almost impossible for this approach to succeed due to the one-way nature of hash algorithms. Hash algorithms are designed so that it is not feasible to reverse the hash to its original value without knowing the specific information of the hash, which provides a robust layer of security for the pseudonymized data.

On the other hand, knowing the hash functions used, we can generate the hash corresponding to a known name and compare it with the values in the pseudonymized DataFrame. This approach shows that the security of the pseudonymized data depends to a large extent on keeping the hash algorithms used confidential.

For the information recovery process, the RIPEMD-160 algorithm followed by SHA-512 was used to generate the hash of the known name "Honora Garbutt".
The generated hash was searched for in the pseudonymized DataFrame to find a match.
If a match was found, the associated data were recovered.

Without knowing the hash functions, no information could be recovered due to the impossibility of reversing the hash without knowing the algorithm used.

In this exercise we can observe the importance of keeping the hash algorithms used in the pseudonymization of data confidential. Although the data are protected by hashing, knowledge of the hash algorithm can allow an attacker to recover sensitive information. Therefore, it is crucial to implement policies and controls that ensure the confidentiality of these methods in order to maintain the security and privacy of the data.










</div>



**Exercise 3:** Dictionary attack — the attacker only knows the workers' country and builds candidate names from lists of common first and last names.

In [14]:
lista_nombres_comunes = ["Daisy", "Philip", "Wilfred", "Honora", "Gale", "Jack", "Roland", "Mary"]
lista_apellidos_comunes = ["Garry", "Neal", "Spooner", "Wade", "Home", "Garbutt", "Brigham", "Wite"]

**3.1** — Knowing the hash functions, recover a person's age, zip code and absences; report the cost in number of hashes computed.

In [15]:
# Answer

# We generate all the possible full names
nombres_completos = [f"{nombre} {apellido}" for nombre, apellido in itertools.product(lista_nombres_comunes, lista_apellidos_comunes)]  # Combines all the common first names and last names into a list of full names

# Number of hashes computed
num_hashes_calculados = 0  # Initializes the counter of hashes computed

# Search for matches in the pseudonymized DataFrame
for nombre_completo in nombres_completos:  # Iterates over all the generated full names
    hashed_name = double_hash(nombre_completo)  # Computes the double hash of the full name
    num_hashes_calculados += 1  # Increments the counter of hashes computed
    matched_row = pseudonymized_workers_df[pseudonymized_workers_df['name'] == hashed_name]  # Looks for the hash in the pseudonymized DataFrame
    if not matched_row.empty:  # If a match is found
        print("Información recuperada:", matched_row[['age', 'zip_code', 'num_absences']].to_dict('records'))  # Prints the recovered information (age, zip code, number of absences)

print("Número total de hashes calculados:", num_hashes_calculados)  # Prints the total number of hashes computed

Información recuperada: [{'age': 58, 'zip_code': '33c896689a7f1bb8207e12fa3eeb8698cf986d7b681548ef17740536f14a1c8c008902e86b1448da40ba2969c5fea02db1040176aef0a11740bd7113db87aff6', 'num_absences': 46}]
Información recuperada: [{'age': 43, 'zip_code': '458aba1e7615af32248bd800e2ef90198b4e52dba4be7b2ca2bf1f8fd2015be70d299c30d32a49e88aa4e26dbe248e2dfedb2e53cd25b71a15ac072a73b0ab46', 'num_absences': 37}]
Información recuperada: [{'age': 50, 'zip_code': '6c760336060d569a298f56111c3014551c70d239b7d96ed062fa66151b9aad6d9af27dca0d234d5a98f31f495fdd74c2c415b0cba5e965b1b106e0d28623e344', 'num_absences': 42}]
Número total de hashes calculados: 64


<div class="alert alert-block alert-success">
Justification:

This method is effective because we combine knowledge, regarding the hash functions and the common names, to carry out a dictionary attack. Its effectiveness depends on the quality of the list of names and the accuracy of the known hash function. Although this method can be costly in computational terms, if the list is large, it is a viable way to try to de-anonymize information if the hash algorithm is known.



</div>

**3.2** — Same attack when unsure whether RIPEMD160+SHA512 or MD5+SHA256 was used; report the cost.

In [16]:
# Answer

def md5_hash(text):  # Defines a function to compute the MD5 hash of a text
    h = hashlib.md5()  # Creates a new MD5 hash object
    h.update(text.encode('utf-8'))  # Updates the hash object with the UTF-8 encoded text
    return h.hexdigest()  # Returns the computed hash in hexadecimal format

def sha256_hash(text):  # Defines a function to compute the SHA-256 hash of a text
    h = hashlib.sha256()  # Creates a new SHA-256 hash object
    h.update(text.encode('utf-8'))  # Updates the hash object with the UTF-8 encoded text
    return h.hexdigest()  # Returns the computed hash in hexadecimal format

def double_hash_md5_sha256(text):  # Defines a function to apply two consecutive hashes (MD5 and SHA-256) to a text
    md5_result = md5_hash(text)  # Computes the MD5 hash of the text
    sha256_result = sha256_hash(md5_result)  # Computes the SHA-256 hash of the previous result
    return sha256_result  # Returns the final result of the double hash

# Number of hashes computed
num_hashes_calculados = 0  # Initializes the counter of hashes computed

# Search for matches in the pseudonymized DataFrame for both algorithms
for nombre_completo in nombres_completos:  # Iterates over all the generated full names
    hashed_name_ripe_sha = double_hash(nombre_completo)  # Computes the double hash (RIPEMD-160 + SHA-512) of the full name
    hashed_name_md5_sha256 = double_hash_md5_sha256(nombre_completo)  # Computes the double hash (MD5 + SHA-256) of the full name
    num_hashes_calculados += 2  # Each name is hashed twice, increments the counter by 2
    matched_row_ripe_sha = pseudonymized_workers_df[pseudonymized_workers_df['name'] == hashed_name_ripe_sha]  # Looks for the hash (RIPEMD-160 + SHA-512) in the pseudonymized DataFrame
    matched_row_md5_sha256 = pseudonymized_workers_df[pseudonymized_workers_df['name'] == hashed_name_md5_sha256]  # Looks for the hash (MD5 + SHA-256) in the pseudonymized DataFrame

    if not matched_row_ripe_sha.empty:  # If a match is found with the hash (RIPEMD-160 + SHA-512)
        print("Información recuperada con RIPEMD160 + SHA512:", matched_row_ripe_sha[['age', 'zip_code', 'num_absences']].to_dict('records'))  # Prints the recovered information
    if not matched_row_md5_sha256.empty:  # If a match is found with the hash (MD5 + SHA-256)
        print("Información recuperada con MD5 + SHA256:", matched_row_md5_sha256[['age', 'zip_code', 'num_absences']].to_dict('records'))  # Prints the recovered information

print("Número total de hashes calculados:", num_hashes_calculados)  # Prints the total number of hashes computed

Información recuperada con RIPEMD160 + SHA512: [{'age': 58, 'zip_code': '33c896689a7f1bb8207e12fa3eeb8698cf986d7b681548ef17740536f14a1c8c008902e86b1448da40ba2969c5fea02db1040176aef0a11740bd7113db87aff6', 'num_absences': 46}]
Información recuperada con RIPEMD160 + SHA512: [{'age': 43, 'zip_code': '458aba1e7615af32248bd800e2ef90198b4e52dba4be7b2ca2bf1f8fd2015be70d299c30d32a49e88aa4e26dbe248e2dfedb2e53cd25b71a15ac072a73b0ab46', 'num_absences': 37}]
Información recuperada con RIPEMD160 + SHA512: [{'age': 50, 'zip_code': '6c760336060d569a298f56111c3014551c70d239b7d96ed062fa66151b9aad6d9af27dca0d234d5a98f31f495fdd74c2c415b0cba5e965b1b106e0d28623e344', 'num_absences': 42}]
Número total de hashes calculados: 128


<div class="alert alert-block alert-success">
Justification:


If we are in doubt between two combinations of hash algorithms (RIPEMD-160 + SHA-512 or MD5 + SHA-256), we can implement both and look for matches in the DataFrame. This approach doubles the number of hashes computed, but it is still viable. The cost of the attack is equal to the number of combinations of first names and last names multiplied by two (to cover both combinations of hash algorithms).


</div>

**Exercise 4:** Propose and implement a scheme (per-record salts) that defeats both previous attacks, and demonstrate that it does.

In [17]:
# Answer
import os

# Hash functions with salt
def generate_salt():                         # Generates a unique salt
    return os.urandom(16).hex()              # Returns the salt in hexadecimal format

def hash_with_salt(text, salt):              # Applies the hash to a text combined with a salt
    combined = text + salt                   # Combines the original text with the salt
    ripemd160_result = ripemd160_hash(combined)  # Computes the RIPEMD-160 hash of the combined text
    sha512_result = sha512_hash(ripemd160_result) # Computes the SHA-512 hash of the previous result
    return sha512_result                     # Returns the final result of the double hash

# Original hash functions
def ripemd160_hash(text):                    # Defines a function to compute the RIPEMD-160 hash of a text
    h = RIPEMD160.new()                      # Creates a new RIPEMD-160 hash object
    h.update(text.encode('utf-8'))           # Updates the hash object with the UTF-8 encoded text
    return h.hexdigest()                     # Returns the computed hash in hexadecimal format

def sha512_hash(text):                       # Defines a function to compute the SHA-512 hash of a text
    h = hashlib.sha512()                     # Creates a new SHA-512 hash object
    h.update(text.encode('utf-8'))           # Updates the hash object with the UTF-8 encoded text
    return h.hexdigest()                     # Returns the computed hash in hexadecimal format

# Pseudonymization with salt
def pseudonymize_with_salt(df):              # Defines a function to pseudonymize identifiers with salt
    salts = []                               # Initializes a list to store the salts
    for i, row in df.iterrows():             # Iterates over each row of the DataFrame
        salt = generate_salt()               # Generates a unique salt for each row
        salts.append(salt)                   # Stores the salt in the list
        df.at[i, 'id'] = hash_with_salt(str(row['id']), salt)  # Applies the salted hash to the ID
        df.at[i, 'name'] = hash_with_salt(row['name'], salt)  # Applies the salted hash to the name
        df.at[i, 'zip_code'] = hash_with_salt(str(row['zip_code']), salt)  # Applies the salted hash to the zip code
    df['salt'] = salts                       # Adds the salts column to the DataFrame
    return df                                # Returns the pseudonymized DataFrame

# We apply pseudonymization with salt
pseudonymized_workers_df = pseudonymize_with_salt(df_workers_original)  # Applies the pseudonymization with salt to the original DataFrame

# We save the DataFrame with the hashes and salts
pseudonymized_workers_df.head()              # Shows the first rows of the pseudonymized DataFrame


,id,name,sex,age,zip_code,num_absences,absences,insurance,salt
0,d7ec9551ece573219b015b1965721dcf4f5085e08fa9e2...,c05039392e0ba7e947afb0d0fd5b7492804059c2e6f855...,M,27,0f324e09a7cd4dc685fd65378a689e1330150818bace83...,23,"[2-7-239554,5-8-205917,4-12-261306,6-4-326452,...",1,aa9f47c78c153779f2ff0551fbee2263
1,04a029f3d1d9710374de57647891bd67d30ea2db9250b6...,732280a2961b2fa5b7226b06043cd49f8912eec02c87fc...,M,28,e6291f968fc370416c6bf9d46f9812ce5ad0db06786124...,6,"[5-8-205917,2-8-205917,6-8-205917,2-4-326452,2...",1,a51ade9e69d95041e1de67f97a580285
2,c6f53a43ee6e7bac45824625403b03bdc83e3902f4c1ab...,f3bfb4d22762d6f12578578573c0d3b3969f41f61c4c4c...,M,29,55d8efe63b5cd557720068e8be97588ac2d3bfaa8fff21...,113,"[4-7-239554,6-7-239554,4-7-239554,4-7-239554,6...",0,1a3cd923f45b77df9d1acaffaed4a544
3,28de6ff98a8314aebefb806aef1c8b3f5a63067079ddbd...,eb1b61dc231697589fa73edf9904c389ebb198a60f7bce...,M,30,dade35875d6b83ccb9a433fc3239019d6750579870b0e6...,1,[3-0-271219],0,4083519e2a4c10ee7ca9a753d92c53a3
4,4fc3e1a59e9b1321e91304311050ae94337836e91eb3b2...,69823e8ae1bee1a4c35ea8031821cc9db4d5b01d17ed23...,F,31,4a2ac7f279e6ce943ad425633aea058e0f0c4a1c814a78...,19,"[3-9-241476,5-9-294217,4-9-294217,5-9-294217,4...",0,2e790b72c5988b8a3e7ef73aba4e8d09


In [18]:
# Check of attack 1
# Attempt a dictionary attack without knowing the salt
for nombre_completo in lista_nombres_comunes:         # Iterates over the list of common names
    hashed_name = double_hash(nombre_completo)        # Computes the double hash of the name
    matched_row = pseudonymized_workers_df[pseudonymized_workers_df['name'] == hashed_name]  # Looks for the hash in the pseudonymized DataFrame
    if not matched_row.empty:                # If a match is found
        print("Información recuperada:", matched_row[['age', 'zip_code', 'num_absences']].to_dict('records'))  # Prints the recovered information

print("Ataque sin conocer el salt no tuvo éxito.")  # Message indicating that the attack was not successful

Ataque sin conocer el salt no tuvo éxito.


In [19]:
# Check of attack 2
# We attempt a dictionary attack without knowing the salts
for nombre_completo in nombres_completos:    # Iterates over all the generated full names
    hashed_name = double_hash_md5_sha256(nombre_completo)  # Computes the double hash of the full name
    matched_row = pseudonymized_workers_df[pseudonymized_workers_df['name'] == hashed_name]  # Looks for the hash in the pseudonymized DataFrame
    if not matched_row.empty:                # If a match is found
        print("Información recuperada:", matched_row[['age', 'zip_code', 'num_absences']].to_dict('records'))  # Prints the recovered information

print("Ataque sin conocer los salts no tuvo éxito.")  # Message indicating that the attack was not successful

Ataque sin conocer los salts no tuvo éxito.


<div class="alert alert-block alert-success">
Justification:

By adding a unique salt to each entry, it is ensured that even if two entries have the same value, their hashes will be different.

Combining the salt with the values before hashing increases the entropy, making dictionary and brute-force attacks much less effective.

Even if an attacker knows the hash algorithms used, without the salts, they will not be able to generate the same hashes that are stored in the DataFrame.

Therefore, this solution shows that incorporating salts into the hashing process provides an additional layer of security, effectively protecting against dictionary and brute-force attacks.

</div>

**Exercise 5:** Encrypt the `absences` attribute with three different cryptosystems and compare the properties of each solution.

**5.1** — Encrypt the absences with RSA using the given public key (`df_workers_rsa`).

In [20]:
public_key_rsa = [15764585266, 15648952315]

In [21]:
# Answer
n, e = public_key_rsa                                      # We assign the values of the RSA public key (n, e)

def rsa_encrypt(message, n, e):                            # We define the function to encrypt a message using RSA
    """Cifra un mensaje usando RSA"""
    return pow(message, e, n)                              # We apply RSA encryption: message^e mod n

# We create a copy of the dataframe for RSA encryption
df_workers_rsa = df_workers.copy()                         # We create a copy of the original dataframe

# We encrypt the absences
def encrypt_absences_rsa(absences):                        # We define the function to encrypt the absences
    absences = absences.replace("[", "").replace("]", "").replace("-", "").replace(" ", "")  # We remove square brackets, hyphens and spaces
    encrypted_absences = [rsa_encrypt(int(absence), n, e) for absence in absences.split(',')]  # We encrypt each absence individually
    return encrypted_absences                              # We return the encrypted absences

df_workers_rsa['absences'] = df_workers_rsa['absences'].apply(encrypt_absences_rsa)  # We apply RSA encryption to the absences column

# We show the dataframe with encrypted absences
df_workers_rsa.head()                                      # We show the first rows of the dataframe encrypted with RSA


,id,name,sex,age,zip_code,num_absences,absences,insurance
0,1,Dwain Hathaway,M,27,8507,23,"[13548500888, 5103837441, 11430239684, 7964062...",1
1,2,Garnett Wolf,M,28,28009,6,"[5103837441, 5528809277, 13765257715, 25924984...",1
2,3,Roy Horsfall,M,29,39011,113,"[11000590288, 9074181250, 11000590288, 1100059...",0
3,4,Kory Easton,M,30,28036,1,[2993075967],0
4,5,Noel Spurling,F,31,49326,19,"[7840850750, 12452260183, 10766506889, 1245226...",0


**5.2** — Encrypt the absences with ElGamal using the given public key (`df_workers_elgamal`).

In [22]:
public_key_elgamal = (2481101477, 747698648, 249871017) # (p, alpha, beta)

In [23]:
from Crypto.PublicKey import ElGamal                       # We import the library to handle ElGamal keys
from Crypto.Random import random                           # We import the library for random number generation
from Crypto.Util.number import bytes_to_long, long_to_bytes  # We import utilities to convert between bytes and long integers

# ElGamal public key
p, alpha, beta = public_key_elgamal                        # We unpack the values of the public key

def elgamal_encrypt(message, p, alpha, beta):              # We define the function to encrypt a message using ElGamal
    """Cifra un mensaje usando ElGamal"""
    k = random.StrongRandom().randint(1, p-2)              # We generate a random number k
    gamma = pow(alpha, k, p)                               # We compute gamma = alpha^k mod p
    delta = (message * pow(beta, k, p)) % p                # We compute delta = message * beta^k mod p
    return (gamma, delta)                                  # We return the pair (gamma, delta)

# We create a copy of the dataframe for ElGamal encryption
df_workers_elgamal = df_workers.copy()                     # We create a copy of the original dataframe

# We encrypt the absences
def encrypt_absences_elgamal(absences):                    # We define the function to encrypt the absences
    absences = absences.replace("[", "").replace("]", "").replace("-", "").replace(" ", "")  # We remove square brackets, hyphens and spaces
    encrypted_absences = [elgamal_encrypt(int(absence), p, alpha, beta) for absence in absences.split(',')]  # We encrypt each absence individually
    return encrypted_absences                              # We return the encrypted absences

df_workers_elgamal['absences'] = df_workers_elgamal['absences'].apply(encrypt_absences_elgamal)  # We apply ElGamal encryption to the absences column

# We show the dataframe with encrypted absences
df_workers_elgamal.head()                                  # We show the first rows of the dataframe encrypted with ElGamal


,id,name,sex,age,zip_code,num_absences,absences,insurance
0,1,Dwain Hathaway,M,27,8507,23,"[(1275726465, 176980684), (10317241, 809079895...",1
1,2,Garnett Wolf,M,28,28009,6,"[(1595876677, 482603945), (367366088, 18609196...",1
2,3,Roy Horsfall,M,29,39011,113,"[(2257279928, 129977756), (2338507814, 1803158...",0
3,4,Kory Easton,M,30,28036,1,"[(339126772, 261528839)]",0
4,5,Noel Spurling,F,31,49326,19,"[(2401565295, 455947838), (1052050456, 2149880...",0


**5.3** — Encrypt the absences with the A5/1 stream cipher initialized from the given register states (`df_workers_a5`).

In [24]:
initial_state_0 = [0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1]
initial_state_1 = [1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1]
initial_state_2 = [1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1]

In [25]:
class A5_1:                                                # We define the A5/1 class for stream encryption
    def __init__(self, initial_state_0, initial_state_1, initial_state_2):
        # Initialize registers with the provided initial states
        self.R1 = initial_state_0                          # Register R1 with the initial state
        self.R2 = initial_state_1                          # Register R2 with the initial state
        self.R3 = initial_state_2                          # Register R3 with the initial state

    def majority(self, x, y, z):                           # Computes the majority bit among x, y and z
        return (x & y) | (x & z) | (y & z)

    def step(self):                                        # Performs one step of the A5/1 algorithm
        maj = self.majority(self.R1[8], self.R2[10], self.R3[10])  # Computes the majority bit
        if self.R1[8] == maj:                              # If the majority bit matches R1[8], R1 is shifted
            self.R1 = [self.R1[18]] + self.R1[:18]
        if self.R2[10] == maj:                             # If the majority bit matches R2[10], R2 is shifted
            self.R2 = [self.R2[21]] + self.R2[:21]
        if self.R3[10] == maj:                             # If the majority bit matches R3[10], R3 is shifted
            self.R3 = [self.R3[22]] + self.R3[:22]

    def get_key_stream(self, length):                      # Generates a key stream of the given length
        key_stream = []                                    # Initializes the key stream
        for _ in range(length):                            # Repeats until the required length is generated
            self.step()                                    # Performs one step of the algorithm
            key_bit = self.R1[-1] ^ self.R2[-1] ^ self.R3[-1]  # Computes the key bit as the XOR of the last bits of the registers
            key_stream.append(key_bit)                     # Appends the key bit to the stream
        return key_stream                                  # Returns the key stream

    def encrypt(self, message):                            # Encrypts a message using A5/1
        key_stream = self.get_key_stream(len(message))     # Generates a key stream of the same length as the message
        encrypted_message = [m ^ k for m, k in zip(message, key_stream)]  # Encrypts each bit of the message with the key stream
        return encrypted_message                           # Returns the encrypted message

# We create a copy of the dataframe for A5/1 encryption
df_workers_a5 = df_workers.copy()                          # We create a copy of the original dataframe

# We encrypt the absences
def encrypt_absences_a5(absences):                         # We define the function to encrypt the absences
    a5 = A5_1(initial_state_0, initial_state_1, initial_state_2)  # We initialize the A5/1 class with the initial states
    absences = absences.replace("[", "").replace("]", "").replace("-", "").replace(" ", "")  # We remove square brackets, hyphens and spaces
    absences_bits = [int(bit) for absence in absences.split(',') for bit in bin(int(absence))[2:].zfill(24)]  # We convert the absences into bits
    encrypted_absences_bits = a5.encrypt(absences_bits)    # We encrypt the bits of the absences using A5/1
    encrypted_absences = ''.join(str(bit) for bit in encrypted_absences_bits)  # We convert the encrypted bits into a string
    return encrypted_absences                              # We return the encrypted absences

df_workers_a5['absences'] = df_workers_a5['absences'].apply(encrypt_absences_a5)  # We apply A5/1 encryption to the absences column

# We show the dataframe with encrypted absences
df_workers_a5.head()                                       # We show the first rows of the dataframe encrypted with A5/1

,id,name,sex,age,zip_code,num_absences,absences,insurance
0,1,Dwain Hathaway,M,27,8507,23,1111000001010101001000100100000000100111011001...,1
1,2,Garnett Wolf,M,28,28009,6,1110000110001110110101000101101011101111110101...,1
2,3,Roy Horsfall,M,29,39011,113,1000101110110011000000111000111100101100010010...,0
3,4,Kory Easton,M,30,28036,1,1101100101110100000110101,0
4,5,Noel Spurling,F,31,49326,19,1010101000110110101000100001011110101111100111...,0


**Exercise 6:** What can an attacker holding only the ciphertext infer under each scheme? **6.1** — Can the number of absences per record be deduced?

In [26]:
# RSA answer
# Example of a list encrypted with RSA
encrypted_absences_rsa = df_workers_rsa.iloc[0]['absences']  # Gets the encrypted absences of the first record
num_absences_rsa = len(encrypted_absences_rsa)  # Counts the number of encrypted absences
num_absences_rsa  # Returns the number of absences

23

<div class="alert alert-block alert-success">
Justification:

For RSA, each absence is encrypted individually. Therefore, the number of absences can be deduced directly from the number of encrypted elements in the list of absences.

In this case, the length of the "list encrypted_absences_rsa" directly indicates the number of absences. This approach is possible because RSA encryption does not alter the number of elements in the list, it only encrypts each one individually.


</div>

In [27]:
# ElGamal answer
# Example of a list encrypted with ElGamal
encrypted_absences_elgamal = df_workers_elgamal.iloc[0]['absences']  # Gets the encrypted absences of the first record
num_absences_elgamal = len(encrypted_absences_elgamal)  # Counts the number of encrypted absences
num_absences_elgamal  # Returns the number of absences

23

<div class="alert alert-block alert-success">
Justification:

Like RSA, ElGamal encrypts each absence individually. Therefore, the number of absences can be deduced from the number of encrypted elements in the list of absences.


The length of the list "encrypted_absences_elgamal" indicates the number of absences. This method is effective because ElGamal encryption also encrypts each element individually without changing the structure of the list.



</div>

In [28]:
# A5/1 answer
# Example of a list encrypted with A5/1
encrypted_absences_a5 = df_workers_a5.iloc[0]['absences']  # Gets the encrypted absences of the first record
len(encrypted_absences_a5)  # Returns the length of the encrypted string

612

<div class="alert alert-block alert-success">
Justification:

For A5/1, the encryption process converts the whole list of absences into a continuous string of bits. This makes it harder to count the absences directly because the structure of the original data is lost in the encryption process.

In this case, it is not possible to count the number of absences directly without knowing the length of the original elements, since everything is converted into a continuous string of bits.


</div>

**6.2** — Given two records, can we tell whether the two people were absent on the same occasion?

In [29]:
# RSA answer

# We compare absences encrypted with RSA
absences1_rsa = set(df_workers_rsa.iloc[0]['absences'])  # Gets the set of encrypted absences of the first record
absences2_rsa = set(df_workers_rsa.iloc[1]['absences'])  # Gets the set of encrypted absences of the second record
common_absences_rsa = absences1_rsa.intersection(absences2_rsa)  # Finds the common absences
common_absences_rsa  # Returns the common absences

{5103837441}

<div class="alert alert-block alert-success">
Justification:

For RSA, comparing absences between different records can be done by directly comparing the encrypted elements.

Being deterministic, it encrypts each element individually, so we can directly compare the encrypted absences to find matches.


</div>

In [30]:
# ElGamal answer

# We compare absences encrypted with ElGamal
absences1_elgamal = set(df_workers_elgamal.iloc[0]['absences'])  # Gets the set of encrypted absences of the first record
absences2_elgamal = set(df_workers_elgamal.iloc[1]['absences'])  # Gets the set of encrypted absences of the second record
common_absences_elgamal = absences1_elgamal.intersection(absences2_elgamal)  # Finds the common absences
common_absences_elgamal  # Returns the common absences

set()

<div class="alert alert-block alert-success">
Justification:

For ElGamal, each absence is encrypted differently even if the original values are the same, due to the probabilistic nature of the encryption.

Since it produces a different ciphertext for the same original value, it is not possible to find matches directly by comparing the encrypted elements.


</div>

In [31]:
# A5/1 answer
# We compare absences encrypted with A5/1
absences1_a5 = df_workers_a5.iloc[0]['absences']  # Gets the encrypted absences of the first record
absences2_a5 = df_workers_a5.iloc[1]['absences']  # Gets the encrypted absences of the second record

<div class="alert alert-block alert-success">
Justification:

For A5/1, comparing absences between different records is extremely difficult due to the nature of the stream cipher, which converts the whole list into a continuous string of bits.

A5/1 mixes all the bits of the absences, so it is not possible to determine whether there are matching absences by directly comparing the encrypted data.



</div>

**6.3** — Can any plaintext absence value be recovered from the encrypted dataset alone?

In [32]:
# RSA answer
# We try to deduce absences using a dictionary attack
# Let us suppose that we know the public key
n, e = public_key_rsa

# Function to try to decrypt absences
def try_decrypt_rsa(encrypted_absences):
    possible_absences = []
    for encrypted in encrypted_absences:
        for absence in range(0, 1000000):  # We assume that the value of the absences is within this range
            if pow(absence, e, n) == encrypted:
                possible_absences.append(absence)
                break
    return possible_absences

# We try with one record
encrypted_absences_rsa = df_workers_rsa.iloc[0]['absences']  # Gets the encrypted absences of the first record
decrypted_absences_rsa = try_decrypt_rsa(encrypted_absences_rsa)  # Tries to decrypt the absences
decrypted_absences_rsa  # Returns the decrypted absences

[]

<div class="alert alert-block alert-success">
Justification:

For RSA, if we know the ciphertext and the public key, we can attempt a dictionary attack to deduce the absences.

If the values of the absences are within a limited range, we can attempt a dictionary attack to deduce the original values.


</div>

In [33]:
# ElGamal answer
# A dictionary attack cannot be carried out with ElGamal due to the probabilistic nature of the encryption.



<div class="alert alert-block alert-success">
Justification:

For ElGamal, due to its probabilistic nature, a dictionary attack is not feasible without the private key.


</div>

In [34]:
# A5/1 answer

# The original absences cannot be deduced without knowing the initial key for A5/1.


<div class="alert alert-block alert-success">
Justification:

For A5/1, without knowing the initial key, it is not possible to deduce the original work absences.



</div>

**Exercise 7:** Check whether the predicted binary attribute `insurance` satisfies the independence property with respect to the protected attribute `sex` (chi-squared test).

In [35]:
# Answer
from scipy.stats import chi2_contingency

# We create the contingency table
contingency_table = pd.crosstab(df_workers['sex'], df_workers['insurance'])  # Creates a contingency table with the values of 'sex' and 'insurance'
print("Tabla de Contingencia:")
print(contingency_table)  # Shows the contingency table

# Summary of the data distribution
print("\nResumen de la Distribución de los Datos:")
print(df_workers['sex'].value_counts())  # Shows the data distribution for 'sex'
print(df_workers['insurance'].value_counts())  # Shows the data distribution for 'insurance'

# We perform the chi-squared test
chi2, p, _, _ = chi2_contingency(contingency_table)  # Performs the chi-squared test on the contingency table
print("\nResultados de la Prueba de Chi-Cuadrado:")
print(f"Chi-cuadrado: {chi2}")  # Shows the chi-squared value
print(f"Valor p: {p}")  # Shows the p-value

Tabla de Contingencia:
insurance  0  1
sex            
F          3  8
M          4  7

Resumen de la Distribución de los Datos:
sex
M    11
F    11
Name: count, dtype: int64
insurance
1    15
0     7
Name: count, dtype: int64

Resultados de la Prueba de Chi-Cuadrado:
Chi-cuadrado: 0.0
Valor p: 1.0


<div class="alert alert-block alert-success">
Justification:

Null Hypothesis (H0): there is no relationship between sex and the granting of the insurance (they are independent).

Alternative Hypothesis (H1): there is a relationship between sex and the granting of the insurance (they are not independent).

Since the p-value (1.0) is much greater than 0.05, we do not reject the null hypothesis. This suggests that there is not enough evidence to claim that a significant relationship exists between sex and the granting of the insurance in this dataset. In other words, the granting of the insurance appears to be independent of the person's sex, thus satisfying the independence property with respect to the protected attribute sex.




</div>